代码生成的输入是AST，目标是IR，所以我们接下来要做的工作其实很Naive，即遍历我们的AST，然后不断的吐出我们的IR指令即可，与此同时，每一个IR都会有自己的IR范式，这些功能都可以通过IR的IRBuilder来做到，于是我们的工作就变为了遍历AST，然后使用IRBuilder构建等价的（优化的）IR出来。Python则为这个提供了很方便的一个基类ast.NodeVisitor，我们只需要继承这个基类，然后override visit_XXX即可以完成我们的目的。

请注意跟着学习至此我认为这个并不是完整了解所有的编译器的细节,而只是了解如何通过这样一个Baby Triton的小的编译器来了解mlc领域的相关内容,同时能够以此为切入点来完善自己的知识体系.

In [1]:
import inspect
import ast

def jit(target="cpu"):
    assert target in ["cpu", "gpu"]
    def inner(fn):
        return JIT(fn, target=target)
    return inner

class JIT:
    def __init__(self, fn, target="cpu"):
        self.fn = fn
        self.target = target
    
    def __call__(self, *args, **kwargs):
        fn_src = inspect.getsource(self.fn)
        fn_ast = ast.parse(fn_src)
        print(ast.dump(fn_ast))
        code_generator = CodeGenerator(fn_ast, self.target)
        code_generator.code_gen()

class CodeGenerator(ast.NodeVisitor):
    def __init__(self, fn_ast, target):
        self.fn_ast = fn_ast
        self.target = target
    
    def code_gen(self):
        self.visit(self.fn_ast)

    def visit(self, node):
        print("Visit " + node.__class__.__name__)
        return super().visit(node)

@jit(target="cpu")
def add():
    print("add")

add()

Module(body=[FunctionDef(name='add', args=arguments(posonlyargs=[], args=[], vararg=None, kwonlyargs=[], kw_defaults=[], kwarg=None, defaults=[]), body=[Expr(value=Call(func=Name(id='print', ctx=Load()), args=[Constant(value='add', kind=None)], keywords=[]))], decorator_list=[Call(func=Name(id='jit', ctx=Load()), args=[], keywords=[keyword(arg='target', value=Constant(value='cpu', kind=None))])], returns=None, type_comment=None)], type_ignores=[])
Visit Module
Visit FunctionDef
Visit arguments
Visit Expr
Visit Call
Visit Name
Visit Load
Visit Constant
Visit Call
Visit Name
Visit Load
Visit keyword
Visit Constant


In [14]:
import inspect
import ast
from tvm.script.ir_builder import relax as relax_builder, ir as I, IRBuilder as IB
def jit(target="cpu"):
    assert target in ["cpu", "gpu"]
    def inner(fn):
        return JIT(fn, target=target)
    return inner

class JIT:
    def __init__(self, fn, target="cpu"):
        self.fn = fn
        self.target = target
    
    def __call__(self, *args, **kwargs):
        fn_src = inspect.getsource(self.fn)
        fn_ast = ast.parse(fn_src)
        print(ast.dump(fn_ast))
        code_generator = CodeGenerator(fn_ast, self.target)
        code_generator.code_gen()

class CodeGenerator(ast.NodeVisitor):
    def __init__(self, fn_ast, target):
        self.fn_ast = fn_ast
        self.target = target
        self.ib = IB()
        self.ir_module = None
    
    def code_gen(self):
        with self.ib:
            self.visit(self.fn_ast)
        module = self.ib.get()
        print(module)


    def visit(self, node):
        print("Visit " + node.__class__.__name__)
        return super().visit(node)
    
    def visit_Module(self, node: ast.Module):
        if self.ir_module:
            raise AssertionError("We should have only one module!")
        self.ir_module = I.ir_module()
        with self.ir_module:
            super().generic_visit(node)
    
    def visit_FunctionDef(self, node: ast.FunctionDef):
        super().generic_visit(node)
        # pass
    
    # def generic_visit(self, node: ast.AST):
    #     raise NotImplementedError("Unsupported AST node type: {}".format(type(node).__name__))


@jit(target="cpu")
def add():
    print("add")

add()

Module(body=[FunctionDef(name='add', args=arguments(posonlyargs=[], args=[], vararg=None, kwonlyargs=[], kw_defaults=[], kwarg=None, defaults=[]), body=[Expr(value=Call(func=Name(id='print', ctx=Load()), args=[Constant(value='add', kind=None)], keywords=[]))], decorator_list=[Call(func=Name(id='jit', ctx=Load()), args=[], keywords=[keyword(arg='target', value=Constant(value='cpu', kind=None))])], returns=None, type_comment=None)], type_ignores=[])
Visit Module
Visit FunctionDef
Visit arguments
Visit Expr
Visit Call
Visit Name
Visit Load
Visit Constant
Visit Call
Visit Name
Visit Load
Visit keyword
Visit Constant
# from tvm.script import ir as I

@I.ir_module
class Module:
    pass


In [18]:
import inspect
import IPython
import ast
import IPython.display
from tvm import relax as rx
from tvm.script import relax as R
from tvm.script.ir_builder import relax as relax_builder, ir as I, IRBuilder as IB
def jit(target="cpu"):
    assert target in ["cpu", "gpu"]
    def inner(fn):
        return JIT(fn, target=target)
    return inner

class JIT:
    def __init__(self, fn, target="cpu"):
        self.fn = fn
        self.target = target
    
    def __call__(self, *args, **kwargs):
        fn_src = inspect.getsource(self.fn)
        fn_ast = ast.parse(fn_src)
        print(ast.dump(fn_ast))
        code_generator = CodeGenerator(fn_ast, self.target)
        code_generator.code_gen()

class CodeGenerator(ast.NodeVisitor):
    def __init__(self, fn_ast, target):
        self.fn_ast = fn_ast
        self.target = target
        self.ib = IB()
        self.ir_module = None
        self.entry = None
        self.ret = None
    
    def code_gen(self):
        with self.ib:
            self.visit(self.fn_ast)
        module = self.ib.get()
        print(module)

    def visit(self, node):
        print("Visit " + node.__class__.__name__)
        return super().visit(node)
    
    def visit_Module(self, node: ast.Module):
        if self.ir_module:
            raise AssertionError("We should have only one module!")
        self.ir_module = I.ir_module()
        with self.ir_module:
            super().generic_visit(node)
        
    
    def visit_FunctionDef(self, node: ast.FunctionDef):
        fn = relax_builder.function()
        self.entry = node.name
        with fn:
            R.func_name(node.name)
            self._visit_compound_stmt(node.body)

            if self.ret is None:
                R.func_ret_value(rx.ShapeExpr([]))
            else:
                R.func_ret_value(self.ret)
    
    def visit_Pass(self, node: ast.Pass):
        pass

    def _visit_compound_stmt(self, stmts):
        assert isinstance(stmts, (list, tuple))
        for stmt in stmts:
            ret = self.visit(stmt)
            if ret is not None and isinstance(stmt, ast.Return):
                self.ret = ret
            
    
    def generic_visit(self, node: ast.AST):
        raise NotImplementedError("Unsupported AST node type: {}".format(type(node).__name__))


@jit(target="cpu")
def add():
    pass

add()

Module(body=[FunctionDef(name='add', args=arguments(posonlyargs=[], args=[], vararg=None, kwonlyargs=[], kw_defaults=[], kwarg=None, defaults=[]), body=[Pass()], decorator_list=[Call(func=Name(id='jit', ctx=Load()), args=[], keywords=[keyword(arg='target', value=Constant(value='cpu', kind=None))])], returns=None, type_comment=None)], type_ignores=[])
Visit Module
Visit FunctionDef
Visit Pass
# from tvm.script import ir as I
# from tvm.script import relax as R

@I.ir_module
class Module:
    @R.function
    def add() -> R.Shape([]):
        return R.shape([])
